In [0]:
import pandas as pd

# ================================================
# LOAD SILVER TABLES - Filter to 2017-2021 only
# ================================================

# Crime Data - filter before toPandas to reduce memory
camb_df = spark.table('crime_data.silver.silver_cambridgeshire_crime') \
    .filter("year BETWEEN 2017 AND 2021") \
    .toPandas()
print(f"✅ Cambridgeshire: {len(camb_df):,} rows")

mers_df = spark.table('crime_data.silver.silver_merseyside_crime') \
    .filter("year BETWEEN 2017 AND 2021") \
    .toPandas()
print(f"✅ Merseyside: {len(mers_df):,} rows")

nott_df = spark.table('crime_data.silver.silver_nottinghamshire_crime') \
    .filter("year BETWEEN 2017 AND 2021") \
    .toPandas()
print(f"✅ Nottinghamshire: {len(nott_df):,} rows")

avon_df = spark.table('crime_data.silver.silver_avon_and_somerset_crime') \
    .filter("year BETWEEN 2017 AND 2021") \
    .toPandas()
print(f"✅ Avon and Somerset: {len(avon_df):,} rows")

# Extra Data
adi_df = spark.table('crime_data.silver.silver_adi') \
    .filter("year BETWEEN 2017 AND 2021") \
    .toPandas()
print(f"✅ ADI: {len(adi_df):,} rows")

houseprice_df = spark.table('crime_data.silver.silver_houseprice') \
    .filter("year BETWEEN 2017 AND 2021") \
    .toPandas()
print(f"✅ House prices: {len(houseprice_df):,} rows")

In [0]:
# Aggregate the crime data for each police force:
camb_agg = camb_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')
mers_agg = mers_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')
nott_agg = nott_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')
avon_agg = avon_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')

In [0]:
# ================================================
# PIVOT, FILL NULLS AND COMBINE ALL FORCES
# ================================================

# Pivot each force
forces_pivoted = []
for i, df in enumerate([camb_agg, mers_agg, nott_agg, avon_agg]):
    pivoted = df.pivot(
        index=['year', 'month_num', 'force_name', 
               'lsoa_code', 'lsoa_name'],
        columns='crime_type',
        values='total_crimes'
    ).reset_index()
    
    # Fill nulls with 0
    pivoted = pivoted.fillna(0)
    
    # Flatten column names
    pivoted.columns.name = None
    
    forces_pivoted.append(pivoted)
    print(f"✅ Force {i+1} pivoted: {len(pivoted):,} rows")

# Combine all forces into one dataset
crime_agg = pd.concat(forces_pivoted, ignore_index=True)

print(f"\n✅ Combined crime data: {len(crime_agg):,} rows")
print(f"✅ Forces: {crime_agg['force_name'].unique().tolist()}")
print(f"✅ Columns: {crime_agg.columns.tolist()}")

In [0]:
# ================================================
# JOIN CRIME DATA WITH ADI (INNER JOIN)
# ================================================

print(f"Rows before ADI join: {len(crime_agg):,}")

crime_agg = crime_agg.merge(
    adi_df[[
        "lsoa_code", "year",
        "ADI",   # ← keep
        "pop"    # ← keep
        # removed: ADI_norm, deprivation_band, adi_yoy_delta
    ]],
    on=["lsoa_code", "year"],
    how="inner"
)

print(f"✅ Rows after ADI join: {len(crime_agg):,}")

# ================================================
# LEFT JOIN WITH HOUSE PRICES
# ================================================

crime_agg = crime_agg.merge(
    houseprice_df[[
        "lsoa_code", "year", "month_num",
        "median_price"   # ← keep
        # removed: transaction_count, log_median_price, price_band
    ]],
    on=["lsoa_code", "year", "month_num"],
    how="left"
)

print(f"✅ Rows after house price join: {len(crime_agg):,}")

In [0]:
# ================================================
# CLEAN COLUMN NAMES AND EXPORT TO GOLD TABLE
# ================================================

# Convert crime counts to integers
crime_cols = [
    'Anti-social behaviour', 'Bicycle theft', 'Burglary',
    'Criminal damage and arson', 'Drugs', 'Other crime',
    'Other theft', 'Possession of weapons', 'Public order',
    'Robbery', 'Shoplifting', 'Theft from the person',
    'Vehicle crime', 'Violence and sexual offences'
]
crime_agg[crime_cols] = crime_agg[crime_cols].fillna(0).astype(int)
print("✅ Crime counts converted to integers")

# Clean column names
crime_agg.columns = (
    crime_agg.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("&", "and")
    .str.replace("/", "_")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace(",", "")
    .str.replace(";", "")
)

print(f"✅ Final columns: {crime_agg.columns.tolist()}")

# Export to gold table
spark.createDataFrame(crime_agg) \
    .write \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable('crime_data.gold.gold_final_dataset')

print(f"✅ Exported to crime_data.gold.gold_final_dataset")
print(f"✅ Total rows: {len(crime_agg):,}")